# protocol

> Claude Code's stream-json wire protocol: NDJSON transport, control routing, and the deferred-tool bridge

In [ ]:
#| default_exp protocol

Speak Claude Code's stream-json wire protocol directly, with no Agent SDK: `read_msgs` frames NDJSON from a piped process, `mk_tools` turns annotated callables or schema dicts into the tool list to advertise, `hold_result` queues an already-known tool result for Claude to collect, `mcp_dispatch` implements the MCP-shaped JSON-RPC methods the CLI uses to list tools and fetch held results, and `ClaudeProto` is the per-process peer: it matches `control_response`s to pending requests, runs each incoming `control_request` as its own cancellable task, routes `PreToolUse` hook callbacks (a call with a held result is allowed, everything else defers, ending the turn so the caller can execute), answers `control_cancel_request` by cancelling and staying silent, and yields every other message through untouched. `initialize` and `interrupt` are ordinary control requests. Tools are never executed here: the caller owns the tool loop, and the runner in `fastclaude.core` owns the process itself.

In [ ]:
#| export
import asyncio, json, os
from fastcore.utils import *
from fastcore.funccall import get_schema
from fastclaude.session import canon

In [ ]:
from fastcore.test import *
from fastclaude.session import ant_data, sess_dir
from collections import Counter
import shutil, sys, tempfile

## The stream

Run with `--output-format stream-json --verbose --include-partial-messages` and piped stdio, `claude` becomes a headless NDJSON peer: every line of stdout is one JSON message, and stdin accepts JSON messages the same way. The fixture below is one real captured run: a Bash tool call, made against a scratch project. Like the transcript fixtures in `fastclaude.session`, the capture is checked in as package data, and the builder returns immediately when it exists; delete the file to request a fresh capture against the installed CLI.

In [ ]:
stream_path = ant_data.parent/'stream.jsonl'

In [ ]:
async def mk_stream_fixture(path=None):  # chkstyle: ignore-node
    "Capture one live claude run's raw stream-json stdout as the checked-in stream fixture"
    path = Path(path) if path else stream_path
    if path.exists(): return path
    prompt = 'think hard: first work out 17*23-4 in your head, then use the Bash tool to run: echo flux-41.7 . Then reply with exactly the tool output followed by your arithmetic answer.'
    argv = ['claude','-p','--output-format','stream-json','--verbose','--include-partial-messages','--model','sonnet','--allowedTools','Bash']
    td = tempfile.mkdtemp()
    p = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
    out,_ = await p.communicate(prompt.encode())
    assert not p.returncode, f'claude exited {p.returncode}'
    path.write_bytes(out)
    shutil.rmtree(sess_dir(td), ignore_errors=True)
    return path

In [ ]:
await mk_stream_fixture()

A run's stdout mixes several kinds of message. Counting them first shows the shape of a whole conversation before we pick each kind apart:

In [ ]:
evs = dict2obj(stream_path.read_jsonl())
len(evs),Counter(e.type for e in evs)

The same content arrives twice. As the model generates, `stream_event` messages wrap the raw Anthropic SSE stream (`message_start`, `content_block_delta`, ...), token by token, for streaming consumers. Then, as each content block completes, a full `assistant` message event carries the finished block; tool results arrive as full `user` message events. The full events match what Claude writes to the session transcript, record for record (same `uuid`s, same `message` content), so a consumer that wants the finished conversation reads only them and ignores the partials:

In [ ]:
tu_ev = first(e for e in evs if e.type=='assistant' and e.message.content[0].type=='tool_use')
tr_ev = first(e for e in evs if e.type=='user')
test_eq(tr_ev.message.content[0].tool_use_id, tu_ev.message.content[0].id)
deltas = [d.event.delta for d in evs if d.type=='stream_event' and d.event.type=='content_block_delta']
test_eq(json.loads(''.join(d.partial_json for d in deltas if d.type=='input_json_delta')), obj2dict(tu_ev.message.content[0].input))
tu_ev.message.content[0].name, tr_ev.message.content[0].content

The rest is ambient noise a reader must tolerate rather than parse: `system` events (`init` with the session's tools and config, `status`, `thinking_tokens`, and `hook_started`/`hook_response` when the user's own hooks fire, which they do even headless), `rate_limit_event`, `command_lifecycle` and `attachment` records on a resume, and whatever new types later CLI versions add. The protocol layer therefore never enumerates event types: it routes the three control messages it owns and passes everything else through untouched.

## NDJSON framing

Pipes deliver chunks, not lines: one read can return half a message, or three. asyncio's `StreamReader.readline` owns the reassembly, bounded by the `limit` passed when the process is spawned (one claude message can carry megabytes of base64 image, so the runner spawns with a generous limit rather than the 64KB default). `read_msgs` turns the byte stream into decoded messages, skipping blank lines and failing loudly on anything that is not JSON: silently dropping a line would hide a desynced stream.

In [ ]:
#| export
async def read_msgs(
    stream, # An asyncio `StreamReader` of NDJSON, e.g. a claude process's stdout
):
    "Decoded messages from `stream`, one per line; blank lines skip, non-JSON raises, a truncated final fragment drops"
    while line := await stream.readline():
        if not (s := line.strip()): continue
        try: yield json.loads(s)
        except json.JSONDecodeError as e:
            if line.endswith(b'\n'): raise ValueError(f'bad NDJSON line: {s[:200]!r}') from e
            return  # no newline: a producer killed mid-write; the fragment is unrecoverable

A scripted producer replays the captured stream in aggressive 37-byte writes, so nearly every message arrives split across reads. Framing reassembles exactly the events the capture holds:

In [ ]:
emit = f"""import sys
data = open({str(stream_path)!r}, 'rb').read()
for i in range(0, len(data), 37):
    sys.stdout.buffer.write(data[i:i+37])
    sys.stdout.buffer.flush()"""
p = await asyncio.create_subprocess_exec(sys.executable, '-c', emit, stdout=asyncio.subprocess.PIPE, limit=2**25)
got = [m async for m in read_msgs(p.stdout)]
await p.wait()
test_eq(got, list(obj2dict(evs)))
test_eq(got[-1]['type'], 'result')

Blank lines skip, a complete final line missing only its newline still arrives, a fragment truncated mid-write is dropped (only a killed producer leaves one), and a non-JSON line raises rather than desyncing:

In [ ]:
def feedr(b):
    "A `StreamReader` pre-fed with `b`, at EOF"
    r = asyncio.StreamReader()
    r.feed_data(b)
    r.feed_eof()
    return r

test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n\n{"b": 2}'))], [dict(a=1), dict(b=2)])
test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n{"cut": tru'))], [dict(a=1)])
bad = read_msgs(feedr(b'not json\n'))
with expect_fail(ValueError, contains='bad NDJSON'): await bad.__anext__()

## Tool schemas


A tool is a schema to advertise, nothing more: the caller executes it, so this layer never holds a callable. A schema dict carries `name`, `description`, and `inputSchema`. An ordinary annotated Python function is accepted as sugar: `get_schema` derives the same dict from its signature and docstring, and the function itself is never called. `mk_tools` normalizes a mixed list of both forms:


In [ ]:
#| export
def tool_spec(
    t, # An annotated callable, or a schema dict with `name`, `description`, `inputSchema`
):
    "The schema dict for one tool; a callable's schema is derived from its signature, and the callable is never executed"
    return get_schema(t, pname='inputSchema') if callable(t) else t

def mk_tools(
    tools, # Tools in either `tool_spec` form
):
    "The schema list to advertise for `tools`"
    return [tool_spec(t) for t in listify(tools)]

In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

py_schema = dict(name='py', description='Run code in the kernel',
    inputSchema=dict(type='object', properties=dict(code=dict(type='string')), required=['code']))
schemas = mk_tools([flux_meter, py_schema])
test_eq([s['name'] for s in schemas], ['flux_meter','py'])
schemas[0]

## Held results

When a history ends in a tool result, the caller already knows the answer to a call Claude has not yet collected. `hold_result` queues that answer, keyed by the *qualified* tool name (`mcp__<server>__<name>`) and the call's canonical arguments, which is how both the transcript and the hook name the call. The bridge serves it when Claude re-invokes the tool on resume. A queue per key, because the same call can legitimately repeat. `tool_reply` converts an Anthropic `tool_result`'s `content` (a string, or text and image blocks) into the MCP return shape:

In [ ]:
#| export
def tool_reply(content, is_error=False):
    "An MCP tool return value carrying an Anthropic `tool_result`'s `content`"
    if isinstance(content, str): content = [dict(type='text', text=content)]
    def _cvt(b):
        if b.get('type')!='image': return b
        s = b['source']
        return dict(type='image', data=s['data'], mimeType=s['media_type'])
    return dict(content=[_cvt(b) for b in content], isError=is_error)

def _res_key(name, input): return canon([name, canon(dict(input))])

def hold_result(
    held, # The store to add to, a dict of queues
    name, # Qualified tool name, e.g. 'mcp__fastclaude__flux_meter'
    input, # The call's arguments, matched canonically
    content, # The known result: an Anthropic `tool_result`'s `content`
    is_error=False, # Was the result an error?
):
    "Queue a result for the bridge to serve when Claude re-invokes `name` with `input`; returns `held`"
    held.setdefault(_res_key(name, input), []).append(tool_reply(content, is_error))
    return held

In [ ]:
held = hold_result({}, 'mcp__fastclaude__flux_meter', dict(unit='gauss'), 'flux: 41.7 gauss')
ib = dict(type='image', source=dict(type='base64', media_type='image/png', data='AAAA'))
test_eq(tool_reply([ib])['content'], [dict(type='image', data='AAAA', mimeType='image/png')])
held


The JSON-RPC envelope is the same three keys every time, so `jrpc` builds one message from its method, id, and params. A None id makes a notification:

In [ ]:
#| export
def jrpc(
    method, # JSON-RPC method name
    id=None, # Request id; None makes a notification
    **params, # The call's `params`, omitted when empty
):
    "One JSON-RPC message, e.g. what the CLI sends our bridge"
    r = dict(jsonrpc='2.0', id=id, method=method)
    if id is None: r.pop('id')
    if params: r['params'] = params
    return r

In [ ]:
test_eq(jrpc('ping', 2), dict(jsonrpc='2.0', id=2, method='ping'))
jrpc('tools/call', 9, name='flux_meter', arguments=dict(unit='gauss'))

## The MCP-shaped bridge

Claude Code talks to the tools we advertise in MCP's JSON-RPC shapes, nested inside its control protocol (the next section). The dispatcher below implements exactly the methods the CLI uses - `initialize`, `notifications/*`, `ping`, `tools/list`, `tools/call` - and nothing else. It is deliberately not a general MCP implementation, and `fastclaude` does not depend on the `mcp` package.

Nothing executes here. `tools/call` pops the held result for the call and serves it; the `PreToolUse` hook (next section) guarantees that only held calls are allowed through, so an un-held call reaching the bridge means the hook and the store disagree, and it answers as an `isError` result naming the gap rather than hanging Claude:


In [ ]:
#| export
def mcp_dispatch(
    msg, # One JSON-RPC message from the CLI
    schemas, # Tool schemas to advertise, from `mk_tools`
    held, # Held results from `hold_result`, keyed by qualified name and arguments
    server='fastclaude', # Server name reported to the CLI, qualifying the held keys
):
    "The JSON-RPC response for `msg`, or None for a notification"
    m,i = msg.get('method'), msg.get('id')
    def res(r): return dict(jsonrpc='2.0', id=i, result=r)
    if m == 'initialize':
        pv = nested_idx(msg, 'params', 'protocolVersion') or '2024-11-05'
        return res(dict(protocolVersion=pv, capabilities=dict(tools={}), serverInfo=dict(name=server, version='1.0')))
    if m and m.startswith('notifications/'): return None
    if m == 'ping': return res({})
    if m == 'tools/list': return res(dict(tools=schemas))
    if m == 'tools/call':
        nm,args = nested_idx(msg, 'params', 'name'), nested_idx(msg, 'params', 'arguments') or {}
        q = held.get(_res_key(f'mcp__{server}__{nm}', args))
        if not q: return res(dict(content=[dict(type='text', text=f'no held result for {nm}')], isError=True))
        return res(q.pop(0))
    return dict(jsonrpc='2.0', id=i, error=dict(code=-32601, message=f'method not found: {m}'))

The dispatcher is a pure function of one message, so its whole contract can be read off in-memory calls; `disp` binds the schemas and store built above, so each check reads as one call. The handshake echoes the client's protocol version, a notification returns None (the outer control request still gets acknowledged, in the next section), and the tool listing is the schemas exactly as built:


In [ ]:
def disp(method, id=None, **p): return mcp_dispatch(jrpc(method, id, **p), schemas, held)

r = disp('initialize', 1, protocolVersion='2025-06-18')
test_eq(r['result']['protocolVersion'], '2025-06-18')
test_is(disp('notifications/initialized'), None)
test_eq(disp('tools/list', 3)['result']['tools'], schemas)
r


`tools/call` serves the held queue for exactly that call, in order. The bare name in the MCP call is qualified with the server prefix before lookup, matching how `hold_result` keys the store. A second result queued behind the first shows the order:


In [ ]:
def fluxcall(): return disp('tools/call', 9, name='flux_meter', arguments=dict(unit='gauss'))['result']

hold_result(held, 'mcp__fastclaude__flux_meter', dict(unit='gauss'), 'flux: 41.8 gauss')
test_eq(fluxcall()['content'][0]['text'], 'flux: 41.7 gauss')
fluxcall()


The queue is now empty, so the same call again is the un-held case: an `isError` result naming the gap, so Claude sees what happened instead of hanging. An unknown JSON-RPC method is the protocol's own error:


In [ ]:
r = fluxcall()
test_eq((r['isError'], r['content'][0]['text']), (True, 'no held result for flux_meter'))
test_eq(disp('resources/list', 4)['error']['code'], -32601)


## Control routing

The three envelope shapes are one-liners, used by the peer below, by test peers, and by anything else that speaks the wire:

In [ ]:
#| export
def ctrl_req(rid, **req):
    "A `control_request` envelope"
    return dict(type='control_request', request_id=rid, request=req)

def ctrl_ok(rid, **resp):
    "A success `control_response` envelope"
    return dict(type='control_response', response=dict(subtype='success', request_id=rid, response=resp))

def ctrl_err(rid, error):
    "An error `control_response` envelope"
    return dict(type='control_response', response=dict(subtype='error', request_id=rid, error=str(error)))

Control traffic rides the same NDJSON stream as everything else, as three message types. A `control_request` carries a `request_id` and a `request` dict, in both directions: Claude sends one to reach our bridge (`subtype: mcp_message`, wrapping one JSON-RPC message) or our hook (`subtype: hook_callback`, carrying the pending call), and we send one for the handshake (`subtype: initialize`) or a native interrupt. A `control_response` answers one by id. A `control_cancel_request` tells us Claude has abandoned a request it made: the handler is cancelled, and no response is written for it.

The hook is how the turn ends at a tool call. `initialize` registers one `PreToolUse` matcher over the qualified tool names; when Claude wants to call one, the CLI asks us first, and the answer decides the turn's fate. A call with a held result is allowed, so the bridge can serve it. Everything else defers: the CLI ends the turn with the `tool_use` on record, and the caller executes.

`ClaudeProto` is the peer for one process: it matches responses to pending requests, spawns one tracked task per incoming request so a slow consumer never blocks the stream, and yields every non-control message through untouched. A JSON-RPC notification returns nothing inner, but the outer control request still gets its acknowledgement, or Claude would wait on it forever.


In [ ]:
#| export
class ClaudeProto:
    "Control-protocol peer for one claude process: request matching, hook routing, held-result serving, passthrough events"
    def __init__(self,
        proc, # An asyncio subprocess speaking stream-json on piped stdin/stdout
        tools=None, # Tool schemas to advertise, in either `tool_spec` form
        held=None, # Held results from `hold_result`; calls without one defer
        server='fastclaude', # SDK MCP server name, matching the `--mcp-config` entry
    ):
        self.proc,self.held,self.server = proc,held or {},server
        self.schemas = mk_tools(tools or [])
        self._lock,self._n,self._pending,self._inflight = asyncio.Lock(),0,{},{}

    async def send(self, obj):
        "Write one JSON message to claude's stdin"
        async with self._lock:
            self.proc.stdin.write(json.dumps(obj, ensure_ascii=False).encode()+b'\n')
            await self.proc.stdin.drain()

    async def send_req(self, req, timeout=60):
        "Send a control request, await its response by id, and return the inner `response` dict"
        self._n += 1
        rid = f'req_{self._n}_{os.urandom(4).hex()}'
        fut = asyncio.get_running_loop().create_future()
        self._pending[rid] = fut
        await self.send(ctrl_req(rid, **req))
        try: return await asyncio.wait_for(fut, timeout)
        finally: self._pending.pop(rid, None)

    async def initialize(self, timeout=120):
        "The control-protocol handshake, registering the defer hook when tools are advertised"
        hooks = None
        if self.schemas:
            matcher = '|'.join(f"mcp__{self.server}__{s['name']}" for s in self.schemas)
            hooks = dict(PreToolUse=[dict(matcher=matcher, hookCallbackIds=['defer'])])
        return await self.send_req(dict(subtype='initialize', hooks=hooks), timeout)

    async def interrupt(self, timeout=30):
        "Claude's native interrupt: end the current turn, keeping the process alive"
        return await self.send_req(dict(subtype='interrupt'), timeout)

In [ ]:
#| export
@patch
def _resolve(self:ClaudeProto, msg):
    "Complete the pending request a `control_response` answers"
    r = msg.get('response') or {}
    if (fut := self._pending.get(r.get('request_id'))) and not fut.done():
        if r.get('subtype')=='error': fut.set_exception(RuntimeError(r.get('error') or 'control request failed'))
        else: fut.set_result(r.get('response') or {})

@patch
def _route(self:ClaudeProto, req):
    "The `PreToolUse` decision: a call with a held result is allowed, everything else defers"
    inp = req.get('input') or {}
    dec = 'allow' if self.held.get(_res_key(inp.get('tool_name'), inp.get('tool_input') or {})) else 'defer'
    return dict(hookSpecificOutput=dict(hookEventName='PreToolUse', permissionDecision=dec))

@patch
async def _handle(self:ClaudeProto, rid, req):
    "Answer one CLI-originated control request; cancelled handlers answer nothing"
    try:
        st = req.get('subtype')
        if st=='mcp_message': resp = dict(mcp_response=mcp_dispatch(req.get('message') or {}, self.schemas, self.held, self.server) or dict(jsonrpc='2.0', result={}))
        elif st=='hook_callback': resp = self._route(req)
        else: raise ValueError(f"unsupported control request: {st}")
        await self.send(ctrl_ok(rid, **resp))
    except asyncio.CancelledError: raise
    except Exception as e: await self.send(ctrl_err(rid, e))

The read loop ties the pieces together; ending it, from either side, runs `aclose`:

In [ ]:
#| export
@patch
async def events(self:ClaudeProto):
    "Non-control messages from claude, with control traffic routed internally"
    try:
        async for m in read_msgs(self.proc.stdout):
            t = m.get('type')
            if t=='control_response': self._resolve(m)
            elif t=='control_request':
                rid = m.get('request_id')
                task = asyncio.create_task(self._handle(rid, m.get('request') or {}))
                self._inflight[rid] = task
                task.add_done_callback(lambda _,rid=rid: self._inflight.pop(rid, None))
            elif t=='control_cancel_request':
                if task := self._inflight.pop(m.get('request_id'), None): task.cancel()
            else: yield m
    finally: await self.aclose()


In [ ]:
#| export
@patch
async def aclose(self:ClaudeProto):
    "Cancel in-flight handlers and pending requests; the process itself is the runner's to reap"
    for t in list(self._inflight.values()): t.cancel()
    for f in self._pending.values():
        if not f.done(): f.cancel()

A scripted peer in memory exercises the whole contract without a process or a model call. The peer's stdout is a `StreamReader` fed by hand; its stdin is a sink collecting what `ClaudeProto` writes back. The held store carries one result, for the `kf` call:


In [ ]:
class _Sink:
    "Collects written messages; a stand-in for a process stdin"
    def __init__(self): self.msgs = []
    def write(self, b): self.msgs.append(json.loads(b))
    async def drain(self): pass

rdr = asyncio.StreamReader()
peer = AttrDict(stdout=rdr, stdin=_Sink())
proto = ClaudeProto(peer, tools=[flux_meter], held=hold_result({}, 'mcp__fastclaude__flux_meter', dict(unit='kf'), 'flux: 41.7 kf'))
def feed(o): rdr.feed_data(json.dumps(o).encode()+b'\n')


The events consumer drains in its own task, and `hook` feeds one `PreToolUse` callback for our tool:

In [ ]:
out = []
async def drain():
    async for m in proto.events(): out.append(m)
t = asyncio.create_task(drain())
def hook(rid, **inp): feed(ctrl_req(rid, subtype='hook_callback', callback_id='defer', input=dict(tool_name='mcp__fastclaude__flux_meter', tool_input=inp)))


Four fed messages tell the story: a hook callback for an un-held call (defer: this is a turn ending), one for the held call (allow), the `tools/call` that follows the allow (served from the store), and a cancel for a request nobody is handling (ignored). The consumer sees only the result event; control traffic never reaches it:


In [ ]:
hook('h1', unit='gauss')
hook('h2', unit='kf')
feed(ctrl_req('m1', subtype='mcp_message', server_name='fastclaude', message=jrpc('tools/call', 1, name='flux_meter', arguments=dict(unit='kf'))))
feed(dict(type='control_cancel_request', request_id='zz'))
await asyncio.sleep(0.05)
feed(dict(type='result', subtype='success'))
rdr.feed_eof()
await t

The decisions came back keyed by request id: defer for the un-held call, allow for the held one, and the held result itself for the MCP call:


In [ ]:
resp = {m['response']['request_id']: m['response']['response'] for m in peer.stdin.msgs}
test_eq(resp['h1']['hookSpecificOutput']['permissionDecision'], 'defer')
test_eq(resp['h2']['hookSpecificOutput']['permissionDecision'], 'allow')
test_eq(resp['m1']['mcp_response']['result']['content'][0]['text'], 'flux: 41.7 kf')
test_eq([m['type'] for m in out], ['result'])
resp['h1']


## A live handshake

The proof that the peer speaks the real protocol is a real process: spawn `claude` with piped stream-json, shake hands, send one user turn, and read to the terminal result. This is the smallest live exchange - no tools, no resume - and the runner in `core` builds on exactly this sequence. It spends tokens, so it stays out of automated runs:

In [ ]:
#| eval: false
td = tempfile.mkdtemp()
argv = ['claude','--output-format','stream-json','--input-format','stream-json','--verbose','--include-partial-messages','--model','sonnet']
lp = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
lproto = ClaudeProto(lp)
res = []
async def ldrain():
    async for m in lproto.events():
        if m.get('type')=='result': return res.append(m)
lt = asyncio.create_task(ldrain())
(await lproto.initialize()).get('output_style')

In [ ]:
#| eval: false
await lproto.send(dict(type='user', message=dict(role='user', content='Reply with exactly: protocol ok')))
await lt
test('protocol ok', res[0]['result'], in_)
res[0]['result']

In [ ]:
#| eval: false
lp.stdin.close()
await lp.wait()
shutil.rmtree(sess_dir(td), ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()